# First experimentation with Python machine learning code


In [34]:
!pip install langchain_community langchain chromadb pypdf tiktoken

In [35]:
# import libraries
import os
from langchain_community.document_loaders import PyPDFLoader
from openai import OpenAI
import json
import requests # type: ignore

# Test chatgpt first:

In [36]:
# Load the JSON file and extract values
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    API_KEY = config.get("API_KEY") # Loading the API Key
    OPENAI_API_BASE = config.get("OPENAI_API_BASE") # Loading the API Base Url

model_name = "gpt-4o-mini"

# Storing API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize OpenAI client
client = OpenAI()

# Create a chat completion
completion = client.chat.completions.create(
    model= model_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello, how are you. are you alive?"}
    ]
)

# Print the assistant's reply
print(completion.choices[0].message.content)


Hello! I'm just a computer program, so I don't have feelings or life in the way that living beings do. However, I'm here and ready to help you with any questions or information you need. How can I assist you today?


# Load PDF

In [37]:
DOC_PATH = "alphabet_10K_2022.pdf"
CHROMA_PATH = "alphabet_db_name"

# load your pdf doc
loader = PyPDFLoader(DOC_PATH)
pages = loader.load()

In [38]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# split the doc into smaller chunks i.e. chunk_size=500
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(pages)

In [39]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma

# get OpenAI Embedding model
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# embed the chunks as vectors and load them into the database
db_chroma = Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_PATH)

In [48]:
# this is an example of a user question (query)
query = 'what are the top risks mentioned in the document that will affect the future of alphabet?'

docs_chroma = db_chroma.similarity_search_with_score(query, k=10)

# generate and answer
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

# Generate answer with LLM

In [49]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

In [50]:
# Prompt template

PROMPT_TEMPLATE = """
Answer the question based only on the following context: {context}

Answer the question based only on the above context: {question}.

Provide a detailed answer.
Don't justify your answers.
Don't give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

In [51]:
# load retrieved context and user query in the prompt template
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, question=query)
print(prompt)

Human: 
Answer the question based only on the following context: harming our business and reputation.
Concerns about, including the adequacy of, our practices with regard to the collection, use, governance, disclosure, or security of personal data or other data-privacy-related matters, even if unfounded, could harm our business, reputation, financial condition,
and operating results. Our policies and practices may change over time as expectations and regulations regarding privacy and data change.
Table of Contents Alphabet Inc.

harming our business and reputation.
Concerns about, including the adequacy of, our practices with regard to the collection, use, governance, disclosure, or security of personal data or other data-privacy-related matters, even if unfounded, could harm our business, reputation, financial condition,
and operating results. Our policies and practices may change over time as expectations and regulations regarding privacy and data change.
Table of Contents Alphabet I

In [55]:
model = ChatOpenAI(model_name=model_name, openai_api_key=API_KEY, openai_api_base=OPENAI_API_BASE)
response_text = model.predict(prompt)
print(response_text)

The top risks that will affect the future of Alphabet include:

1. Concerns regarding data privacy practices: There are risks related to the adequacy of practices concerning the collection, use, governance, disclosure, or security of personal data. Even unfounded concerns can harm the business, reputation, financial condition, and operating results.

2. Trademark protection: A potential loss of trademark protection for the word "Google" could occur if a different determination is reached, leading to diminished brand value as others may use the term for their own products.

3. Security vulnerabilities: The storage, handling, and transmission of proprietary and sensitive information expose the company to risks due to software bugs, theft, misuse, defects, vulnerabilities, and security breaches.

4. Loss of key personnel: The future success of Alphabet heavily relies on the continued service of key members of the senior management team. The loss of such personnel could hinder the executio

In [56]:
model = ChatOpenAI(openai_api_key=API_KEY)
response_text = model.predict(prompt)
print(response_text)

1. Concerns about privacy practices and data security could harm business and reputation.
2. Losing protection for the trademark "Google" could result in brand diminishment.
3. Risks associated with software bugs, theft, misuse, defects, vulnerabilities, and security breaches.
4. Potential loss of key personnel impacting the execution of business strategy.
5. Lack of visibility over encrypted services and platform activity leading to incidents or activities outside of the company's control.
